## Loading Libraries and Data

##### Loading core libraries: numpy. pandas and scikit-learn libraries for data processing, preprocessing, model pipelining and model evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

##### Loading data files: Reading the created training and testing files. The original files were split into training and validation sets, as shown below. These files were loaded alongside the original training and testing csv files for model training and evaluation.

In [ ]:
dir = "../original_rounds/"
train_data_to_5 = pd.read_csv(f"{dir}rounds_1_5.csv", low_memory = False)
val_data_6 = pd.read_csv(f"{dir}round_6.csv", low_memory = False)

train_data_to_6 = pd.read_csv(f"{dir}rounds_1_6.csv", low_memory = False)
val_data_7 = pd.read_csv(f"{dir}round_7.csv", low_memory = False)

train_data_to_7 = pd.read_csv(f"{dir}rounds_1_7.csv", low_memory = False)
val_data_8 = pd.read_csv(f"{dir}round_8.csv", low_memory = False)

train_data = pd.read_csv(f"{dir}train.csv",low_memory = False)
test_data = pd.read_csv(f"{dir}test.csv", low_memory=False)

## Data Preprocessing 

##### Defining reusable functions: Creating reusable function to seperate numerical and categorical columns in the dataset. These functions implement the preprocessing pipelines required to prepare the data for model training and evaluation.

In [ ]:
def seperate_num_cat(train_data,test_data ):
    data_train = train_data[train_data['employed_status'].notna()].copy()
    output_train = data_train['employed_status'].astype(int)
    data_test = test_data[test_data['employed_status'].notna()].copy()
    output_test = data_test['employed_status'].astype(int)

    numerical = ['age','work_readiness_score','years_unemployed','household_size','total_historical_rounds','tenure_lag','survey_round']
    category= ['gender','province','district','municipality','education_level','has_internet_access','sa_citizen','race','school_quintile','status_broad_lag','employed_lag','status_lag']

    num_columns = [c for c in numerical if c in train_data.columns]
    cat_columns = [c for c in category if c in train_data.columns]

    total_col = num_columns + cat_columns
    dummy_train = data_train[total_col].copy()
    dummy_test = data_test[total_col].copy()

    for x in (dummy_train, dummy_test):
        for c in cat_columns:
            x[c] = x[c].astype(str)

    return output_train, output_test, dummy_train, dummy_test, num_columns, cat_columns

def cat_pipeline():
    return Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Missing')),('oh',OneHotEncoder(handle_unknown='ignore'))])

def num_pipeline():
    return Pipeline([('impute', SimpleImputer(strategy='median')),('scale',StandardScaler())])

def build(num_columns, cat_columns):
    preprocessor = ColumnTransformer([('cat',cat_pipeline(),cat_columns),('num', num_pipeline(),num_columns),])
    return preprocessor

# Testing

##### Creating Baseline Model: Developing Baseline models for both Logistic Regression and Gradient Boosting. Testing begins with rounds 1-5 as training data and rounds 6 as validation data. The AUC score and permutation feature importance are collected and used for the following tests

In [ ]:
train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_5, val_data_6)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

permutation = permutation_importance(logistic_model,dummy_test,test_output,scoring='roc_auc',n_repeats=10,random_state=42)
perm_df = pd.DataFrame({'feature' : dummy_test.columns, 'imp_mean': permutation.importances_mean, 'imp_std': permutation.importances_std})
print(perm_df)

In [ ]:
grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
grad_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {grad_auc:.4f}')

permutation = permutation_importance(grad_boost,dummy_test,test_output,scoring='roc_auc',n_repeats=10,random_state=42)
perm_df = pd.DataFrame({'feature' : dummy_test.columns, 'imp_mean': permutation.importances_mean, 'imp_std': permutation.importances_std})
print(perm_df)

##### Testing without negative importance: Testing both models after removing columns with a negative imp_mean to evaluate if these features improves model performance.

In [ ]:
drop = ['education_level','sa_citizen','race','school_quintile']
train_data_to_5_reduced = train_data_to_5.drop(columns=drop)
val_data_6_reduced = val_data_6.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_5_reduced, val_data_6_reduced)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

In [ ]:
drop = ['municipality','sa_citizen','race','school_quintile']
train_data_to_5_reduced_neg = train_data_to_5.drop(columns=drop)
val_data_6_reduced_neg = val_data_6.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_5_reduced_neg, val_data_6_reduced_neg)
pre = build(num_columns, cat_columns)

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
grad_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {grad_auc:.4f}')

##### Testing without low importance features: Testing both models after the low/negative imp_mean columns to evaluate if these features improves model performance.

In [ ]:
drop = ['education_level','sa_citizen','race','school_quintile','education_level','district','status_broad_lag']
train_data_to_5_reduced_all = train_data_to_5.drop(columns=drop)
val_data_6_reduced_all = val_data_6.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_5_reduced_all, val_data_6_reduced_all)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

In [ ]:
drop = ['municipality','sa_citizen','race','school_quintile']
train_data_to_5_reduced_all = train_data_to_5.drop(columns=drop)
val_data_6_reduced_all = val_data_6.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_5_reduced_all, val_data_6_reduced_all)
pre = build(num_columns, cat_columns)

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
grad_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {grad_auc:.4f}')

### Testing done on following rounds :Repeat of testing process using the same methodology as the previous tests. Training data, rounds 1-6, and validation data, round 7.

In [ ]:
drop = ['education_level','sa_citizen','race','school_quintile','education_level','district','status_broad_lag']
train_data_to_6_reduced= train_data_to_6.drop(columns=drop)
val_data_7_reduced = val_data_7.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_6_reduced, val_data_7_reduced)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

In [ ]:
drop = ['municipality','sa_citizen','race','school_quintile']
train_data_to_6_reduced= train_data_to_6.drop(columns=drop)
val_data_7_reduced = val_data_7.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_6_reduced, val_data_7_reduced)
pre = build(num_columns, cat_columns)

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
grad_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {grad_auc:.4f}')

In [ ]:
train_data_to_7_reduced= train_data_to_7.copy()
val_data_8_reduced = val_data_8.copy()

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_7_reduced, val_data_8_reduced)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

permu2 = permutation_importance(logistic_model,dummy_test,test_output,scoring='roc_auc',n_repeats=10,random_state=42)
perm_df2 = pd.DataFrame({'feature' : dummy_test.columns, 'imp_mean': permu2.importances_mean, 'imp_std': permu2.importances_std})
print(perm_df2)

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
grad_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {grad_auc:.4f}')

permu1 = permutation_importance(grad_boost,dummy_test,test_output,scoring='roc_auc',n_repeats=10,random_state=42)
perm_df1 = pd.DataFrame({'feature' : dummy_test.columns, 'imp_mean': permu1.importances_mean, 'imp_std': permu1.importances_std})
print(perm_df1)

In [ ]:
drop = ['total_historical_rounds','sa_citizen']
train_data_to_7_reduced= train_data_to_7.drop(columns=drop)
val_data_8_reduced = val_data_8.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_7_reduced, val_data_8_reduced)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')


In [ ]:
drop = ['total_historical_rounds','district']
train_data_to_7_reduced = train_data_to_7.drop(columns=drop)
val_data_8_reduced = val_data_8.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_7_reduced, val_data_8_reduced)
pre = build(num_columns, cat_columns)
grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
grad_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {grad_auc:.5f}')

#### Modifying the Numerical and Categorical Seperator function: For the purpose of testing, allowing the test data set to 'exclude' the employed_status column as it has no labels. It replaces the null categorical columns cells with 'Missing' to ensure the data can be processed properly. 

In [ ]:
def seperate_num_cat_test(train_data,test_data):
    data_train = train_data[train_data['employed_status'].notna()].copy()
    output_train = data_train['employed_status'].astype(int)
    data_test = test_data.copy()

    numerical = ['age','work_readiness_score','years_unemployed','household_size','total_historical_rounds','tenure_lag','survey_round']
    category= ['gender','province','district','municipality','education_level','has_internet_access','sa_citizen','race','school_quintile','status_broad_lag','employed_lag','status_lag']

    num_columns = [c for c in numerical if c in train_data.columns]
    cat_columns = [c for c in category if c in train_data.columns]

    total_col = num_columns + cat_columns
    dummy_train = data_train[total_col].copy()
    dummy_test = data_test[total_col].copy()
    for x in (dummy_train, dummy_test):
        for c in cat_columns:
            x[c] = x[c].where(x[c].notna(), 'Missing').astype(str)

    return output_train, dummy_train, dummy_test, num_columns, cat_columns

#### Creating the final model: Training the final model with the variables provided with testing. These models will be submitted to be evaluated.

In [ ]:
drop = ['total_historical_rounds','district']
train_reduced = train_data.drop(columns=drop)
test_reduced = test_data.drop(columns=drop)

train_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat_test(train_reduced, test_reduced)
pre = build(num_columns, cat_columns)

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]

In [ ]:
drop = ['total_historical_rounds','sa_citizen']
trainlog_reduced = train_data.drop(columns=drop)
testlog_reduced = test_data.drop(columns=drop)

train_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat_test(trainlog_reduced, testlog_reduced)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]

In [ ]:
id = test_data['anonymised_id'].copy()
preds = pd.DataFrame({'anonymised_id':id, 'employed_status': grad_val_prob})
preds2 = pd.DataFrame({'anonymised_id':id, 'employed_status': log_model_val_prob})
preds.to_csv('predictions/submission_upgraded_gb.csv', index=False)
preds2.to_csv('predictions/submission_upgraded_log.csv', index=False)